In [1]:
from osgeo import gdal, osr
import numpy as np
import waves
from pathlib import Path
import geopandas as gpd

gdal.UseExceptions()

In [2]:
crs = "EPSG:25833"
lamp_path  = "../niva/tare/LAMpred_Norway.tif"
depth_path = "../niva/tare/feature_norge_dem50_depth_filled.tif"

In [3]:
lamp_ds  = gdal.Open(lamp_path,  gdal.GA_ReadOnly)
depth_ds = gdal.Open(depth_path, gdal.GA_ReadOnly)

print("LAMpred  CRS:", lamp_ds.GetProjection()[:60], "...")
print("LAMpred  size:", lamp_ds.RasterXSize, "x", lamp_ds.RasterYSize)
print("Depth    CRS:", depth_ds.GetProjection()[:60], "...")
print("Depth    size:", depth_ds.RasterXSize, "x", depth_ds.RasterYSize)

LAMpred  CRS: PROJCS["WGS 84 / UTM zone 33N",GEOGCS["WGS 84",DATUM["WGS_19 ...
LAMpred  size: 50984 x 63377
Depth    CRS: PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["Europe ...
Depth    size: 24431 x 30735


In [3]:
base_name = "stortare"
base_path =  waves.paths.TARE_DIR / base_name
base_path.mkdir(exist_ok=True)

In [5]:
# Reproject LAMpred to EPSG:25833, align depth grid, and apply depth mask (≤ -50 m)
lamp_reproj_path = base_path / "LAMpred_Norway_25833.tif"
depth_aligned_path = base_path / "depth_aligned_stortare_25833.tif"
fname = waves.utils.to_filename(
    "stortare-tetthetsklasser", "norge", "2021", crs.split(":")[-1]
)
filtered_path = waves.paths.TARE_DIR / f"{fname}.tif"
vector_raster_path = base_path / "filtered-vector.tif"

filtered_ds = waves.filter.reproject_and_mask_depth(
    src_ds=lamp_ds,
    depth_ds=depth_ds,
    dst_crs=crs,
    reproj_path=lamp_reproj_path,
    depth_aligned_path=depth_aligned_path,
    filtered_path=filtered_path,
    vector_path=vector_raster_path,
)
gt = filtered_ds.GetGeoTransform()
print(f"Filtered size: {filtered_ds.RasterXSize} x {filtered_ds.RasterYSize}")
print(f"Pixel size: {gt[1]:.2f} m x {abs(gt[5]):.2f} m")
filtered_ds = None  # close


Pixels before depth filter: 252,045,561
Pixels after  depth filter: 45,028,384  (17.9% retained)
Written to: /home/kim/work/marine-model-processing/niva/tare/stortare-tetthetsklasser_norge_2021_25833.tif
Vector raster (depth nodata kept): /home/kim/work/marine-model-processing/niva/tare/stortare/filtered-vector.tif
Filtered size: 50984 x 63377
Pixel size: 25.00 m x 25.00 m


In [5]:
skog_path = waves.paths.TARE_DIR / (base_name + "-value.tif")
skog_path_vec = waves.paths.TARE_DIR / (base_name + "-value-vector.tif")

In [ ]:
stortare_ds = gdal.Open(filtered_path, gdal.GA_ReadOnly)

stortare_filtered_ds = waves.filter.apply_value_threshold(
    src_ds=stortare_ds,
    out_path=skog_path,
    min_value=5,
)
stortare_filtered_ds = None  # close

# Value-threshold raster for vector production (depth-nodata pixels kept)

stortare_vec_ds = gdal.Open(str(vector_raster_path), gdal.GA_ReadOnly)
stortare_vec_filtered_ds = waves.filter.apply_value_threshold(
    src_ds=stortare_vec_ds,
    out_path=skog_path_vec,
    min_value=5,
)
stortare_vec_filtered_ds = None  # close

Pixels before value filter: 45,028,384
Pixels after  value filter: 40,113,609  (89.1% retained)
Written to: /home/kim/work/marine-model-processing/niva/tare/stortare-value.tif
Pixels before value filter: 151,371,302
Pixels after  value filter: 146,032,253  (96.5% retained)
Written to: /home/kim/work/marine-model-processing/niva/tare/stortare-value-vector.tif


## Create final class based raster

In [6]:
skog_classified_path =  waves.paths.TARE_DIR /  (waves.utils.to_filename("stortare-skog", "norge", "2021", crs.split(":")[-1]) + ".tif")

In [ ]:
# Class-based raster from fname (depth-nodata masked)
skog_ds = gdal.Open(str(skog_path), gdal.GA_ReadOnly)

# 1. Classify density values → MIDDELS_TETT (3) / TETT (4)
classified_ds = waves.tare.classify_skog(skog_ds, skog_classified_path)
classified_ds = None

# 2. Merge classes 3 & 4 → 1 in the final raster
ds = gdal.Open(str(skog_classified_path), gdal.GA_Update)
band = ds.GetRasterBand(1)
arr = band.ReadAsArray()
arr[(arr == 3) | (arr == 4)] = 1
band.WriteArray(arr)
ds.FlushCache()
ds = None

Classified raster written to: /home/kim/work/marine-model-processing/niva/tare/stortare-skog_norge_2021_25833.tif


# Create vector dataset

In [ ]:
skog_sieved_path     = base_path / (base_name + "-klasser-sieved.tif")
skog_raw_vec_path    = base_path / (base_name + "-raw.gpkg")
skog_clipped_path    = base_path / (base_name + "-land-clipped.gpkg")
skog_final_path      = base_path / (base_name + ".gpkg")

In [ ]:
# Vector pipeline from raster without depth-nodata masking
skog_vec_classified_path = base_path / (base_name + "-klasser-for-vector.tif")
skog_vec_ds = gdal.Open(str(skog_path_vec), gdal.GA_ReadOnly)
vec_classified_ds = waves.tare.classify_skog(skog_vec_ds, skog_vec_classified_path)
vec_classified_ds = None

# 2. Merge classes 3 & 4 → 1 in the final raster
ds = gdal.Open(str(skog_vec_classified_path), gdal.GA_Update)
band = ds.GetRasterBand(1)
arr = band.ReadAsArray()
arr[(arr == 3) | (arr == 4)] = 1
band.WriteArray(arr)
ds.FlushCache()
ds = None

# 2. Sieve filter: remove standalone pixels
waves.filter.sieve_filter(skog_vec_classified_path, skog_sieved_path, threshold=2, connectedness=4)

# 3. Vectorize (no generalisation yet)
waves.tare.vectorize_skog(skog_sieved_path, skog_raw_vec_path)

# 4. Subtract land
gdf_raw = gpd.read_file(skog_raw_vec_path)
gdf_clipped = waves.tare.subtract_land(gdf_raw, waves.paths.LAND)
gdf_clipped.to_file(skog_clipped_path, driver="GPKG", layer="stortare_skog")

# 5. Douglas-Peucker generalisation via GRASS (topology-preserving)
waves.tare.generalize_skog(
    input_gpkg=skog_clipped_path,
    output_gpkg=skog_final_path,
    threshold=waves.tare.SMOOTH_THRESHOLD,
)

Classified raster written to: /home/kim/work/marine-model-processing/niva/tare/stortare/stortare-klasser-for-vector.tif
Copying classified raster...
0...10...20...30...40...50...60...70...80...90...Applying sieve filter (threshold=2, connectedness=4)...
100 - done in 00:00:12.
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:54.
Vectorizing stortare-klasser-sieved.tif (EPSG:25833) …


Importing raster map <raster>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Extracting areas...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Writing areas...
   0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68  72  76  80  84  88  92  96 100
Building topology for vector map <vector@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
Attaching islands...
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  4

Done → /home/kim/work/marine-model-processing/niva/tare/stortare/stortare-raw.gpkg
Subtracting land from 44,050 of 75,771 polygons …
Generalizing stortare-land-clipped.gpkg (threshold=12.0 m) …


Check if OGR layer <stortare_skog> contains polygons...
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
Creating attribute table for layer <stortare_skog>...
Column name <cat> renamed to <cat_>
Importing 72661 features (OGR layer <stortare_skog>)...
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
-----------------------------------------------------
Registering primitives...
-----------------------------------------------------700     800     900    1000    1100    1200    1300    1400
Cleaning polygons
-----------------------------------------------------
Breaking polygons...
Breaking polygons (pass 1: select break points)...
   1   3   5   7   9  11  13  15  17  19  21  23  2

Done → /home/kim/work/marine-model-processing/niva/tare/stortare/stortare.gpkg


         category are written only when -c flag is given.


PosixPath('/home/kim/work/marine-model-processing/niva/tare/stortare/stortare.gpkg')

In [8]:
gdf = gpd.read_file(skog_final_path).explode()

klasse_map_no = {int(k): v["navn_no"] for k, v in waves.tare.KLASSE_INFO_MERGED_STORTARE.items()}
klasse_map_en = {int(k): v["navn_en"] for k, v in waves.tare.KLASSE_INFO_MERGED_STORTARE.items()}
gdf["navn_no"] = gdf["klasse_int"].map(klasse_map_no)
gdf["navn_en"] = gdf["klasse_int"].map(klasse_map_en)


In [14]:
tare_vec = skog_classified_path.with_suffix(".gpkg")
tare_vec.unlink(missing_ok=True)

grunnlinje = gpd.read_file("../aux/grunnlinje_1nautisk.gpkg")
gdf_clipped = gpd.overlay(gdf, grunnlinje, how="intersection")[['klasse_int', 'geometry', 'navn_no', 'navn_en']]
gdf_clipped.to_file(tare_vec, driver="GPKG", layer="stortareskog")